# Scraper Test

Diagnostic notebook for `common/scraping.py`'s `scrape_fragrantica()`. Run
this whenever the scraper seems broken, or after Fragrantica changes their
site, to see exactly which fields are/aren't extracting correctly.

For each test URL below, checks whether each of the 8 fields came back
populated (not `"N/A"`/empty) and prints a pass/fail summary table, plus
the full raw result per URL for manual inspection.


In [1]:
import sys

sys.path.append("../..")  # repo root, so `common` is importable

from common.scraping import scrape_fragrantica


In [2]:
# Known-current Fragrantica perfume pages to test against. Keep this list
# small but real — a URL that no longer resolves to the fragrance you think
# it does (Fragrantica IDs can get reassigned) will produce a misleading
# "failure" that's actually just a stale test URL, not a scraper bug.
TEST_URLS = [
    "https://www.fragrantica.com/perfume/Fragrance-World/Velvet-Rouge-104781.html",
    "https://www.fragrantica.com/perfume/Dior/Sauvage-31861.html",
]


In [3]:
FIELDS = ["name", "gender", "rating", "rating_count", "main_accords", "perfumers", "description"]


def field_ok(value):
    """A field "worked" if it's not the N/A sentinel and not an empty list."""
    if value in ("N/A", None):
        return False
    if isinstance(value, list) and len(value) == 0:
        return False
    return True


results = []
raw_results = {}

for url in TEST_URLS:
    data = scrape_fragrantica(url)
    raw_results[url] = data

    if "error" in data:
        results.append({"url": url, "status": f"ERROR: {data['error']}", **{f: False for f in FIELDS}})
        continue

    row = {"url": url, "status": "fetched"}
    for f in FIELDS:
        row[f] = field_ok(data.get(f))
    results.append(row)


In [4]:
import pandas as pd

summary_df = pd.DataFrame(results)
display(summary_df) if "display" in dir() else print(summary_df.to_string(index=False))

n_urls = len(TEST_URLS)
print()
for f in FIELDS:
    passing = sum(1 for r in results if r.get(f) is True)
    flag = "OK" if passing == n_urls else ("BROKEN" if passing == 0 else "PARTIAL")
    print(f"{f:15s} {passing}/{n_urls} passing  [{flag}]")


,url,status,name,gender,rating,rating_count,main_accords,perfumers,description
0,https://www.fragrantica.com/perfume/Fragrance-...,fetched,True,True,True,True,True,False,True
1,https://www.fragrantica.com/perfume/Dior/Sauva...,fetched,True,True,True,True,True,True,True



name            2/2 passing  [OK]
gender          2/2 passing  [OK]
rating          2/2 passing  [OK]
rating_count    2/2 passing  [OK]
main_accords    2/2 passing  [OK]
perfumers       1/2 passing  [PARTIAL]
description     2/2 passing  [OK]


In [5]:
import json

for url, data in raw_results.items():
    print("===", url)
    print(json.dumps(data, indent=2, ensure_ascii=False))
    print()


=== https://www.fragrantica.com/perfume/Fragrance-World/Velvet-Rouge-104781.html
{
  "url": "https://www.fragrantica.com/perfume/Fragrance-World/Velvet-Rouge-104781.html",
  "name": "Velvet Rouge Fragrance World",
  "gender": "for women and men",
  "rating": 4.23,
  "rating_count": 70,
  "main_accords": [
    "rose",
    "woody",
    "patchouli",
    "earthy",
    "musky",
    "warm spicy",
    "green",
    "floral",
    "powdery",
    "aromatic"
  ],
  "perfumers": [],
  "description": "Velvet Rouge by Fragrance World is a Floral Woody Musk fragrance for women and men. Velvet Rouge was launched in 2023. Top notes are White Tea and Pink Pepper; middle notes are Rose, Iris and Jasmine; base notes are Patchouli, Vetiver and Musk."
}

=== https://www.fragrantica.com/perfume/Dior/Sauvage-31861.html
{
  "url": "https://www.fragrantica.com/perfume/Dior/Sauvage-31861.html",
  "name": "Sauvage Dior",
  "gender": "for men",
  "rating": 3.85,
  "rating_count": 33812,
  "main_accords": [
    "fre